# griffelib smoke test

Smoke test for the `griffelib` distribution (the internal library half of the griffe 2.x split — provides the `_griffe` namespace consumed by the public `griffe` API and the `griffecli` command-line tool).

In [ ]:
# Example 1: Load a stdlib module via griffe's public Loader API.
import griffe

loader = griffe.GriffeLoader()
json_module = loader.load('json')

print(f"Loaded module: {json_module.name}")
print(f"Kind: {json_module.kind}")
print(f"Is module: {json_module.is_module}")
assert json_module.name == 'json'
assert json_module.is_module

In [ ]:
# Example 2: Inspect members of a loaded module.
import griffe

loader = griffe.GriffeLoader()
json_mod = loader.load('json')

members = list(json_mod.members.keys())
print(f"json has {len(members)} top-level members")
print(f"First 10: {members[:10]}")

# Common public API members of json
assert 'loads' in members or 'dumps' in members, f"expected json.loads / json.dumps in {members}"
print("\njson.loads is present in loaded griffe model")

In [ ]:
# Example 3: Parse a small Python source string via griffe.
# temporary_visited_module is a context manager (griffe 2.x API).
import textwrap

import griffe

src = textwrap.dedent('''
    """A tiny demo module."""

    def add(a: int, b: int) -> int:
        """Return the sum of a and b."""
        return a + b

    class Greeter:
        def hello(self, name: str) -> str:
            return f"hi {name}"
''')

with griffe.temporary_visited_module(src) as mod:
    print(f"Module members: {list(mod.members.keys())}")
    assert 'add' in mod.members
    assert 'Greeter' in mod.members
    assert mod.members['add'].is_function
    assert mod.members['Greeter'].is_class

In [ ]:
# Example 4: Inspect function signature parameters via griffe model.
# temporary_visited_module is a context manager (griffe 2.x API).
import textwrap

import griffe

src = textwrap.dedent('''
    def f(x: int, y: str = "hi", *, z: bool = True) -> str:
        return f"{x}-{y}-{z}"
''')

with griffe.temporary_visited_module(src) as mod:
    f = mod['f']
    param_names = [p.name for p in f.parameters]
    print(f"f parameters: {param_names}")
    assert param_names == ['x', 'y', 'z']
    print(f"return annotation: {f.returns}")

In [ ]:
# Example 5: Verify core public symbols exist on the griffe namespace.
import griffe

expected_symbols = ['GriffeLoader', 'Module', 'Class', 'Function', 'Parameter']
missing = [s for s in expected_symbols if not hasattr(griffe, s)]

for s in expected_symbols:
    if hasattr(griffe, s):
        print(f"griffe.{s} -> {getattr(griffe, s)!r}")

assert not missing, f"Missing core symbols: {missing}"
print("\nAll core griffe public symbols present")